In [1]:
import csv
import json
import shutil
from pathlib import Path

# Student Inputs
LAST_NAME = "Asis"
STUDENT_ID = "TUPM-26-1217"
SEED_DIGIT = int(STUDENT_ID[-1])  

# Data Generation Rules
ASSIGNED_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"  
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1               

# Paths
DATASET_PATH = Path("data/user_behavior_dataset.csv")
FILTERED_CSV_PATH = Path("output/filtered_user_behavior.csv")
SUMMARY_JSON_PATH = Path("output/user_behavior_summary.json")
BACKUP_CSV_PATH = Path("backup/filtered_user_behavior_backup.csv")

# Containers
total_records_in_dataset = 0
os_matching_records = []
target_behavior_records = []

total_battery_drain = 0.0
total_data_usage = 0.0

# Read CSV dataset and filter matching records
with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records_in_dataset += 1

        try:
            user_id = row.get("User ID", "").strip()
            device_model = row.get("Device Model", "").strip()
            os_name = row.get("Operating System", "").strip()
            battery_drain_str = row.get("Battery Drain (mAh/day)", "").strip()
            data_usage_str = row.get("Data Usage (MB/day)", "").strip()
            behavior_class_str = row.get("User Behavior Class", "").strip()

            if not (battery_drain_str and data_usage_str and behavior_class_str):
                continue

            battery_drain = float(battery_drain_str)
            data_usage = float(data_usage_str)
            behavior_class = int(behavior_class_str)

            if os_name.lower() == ASSIGNED_OS.lower():
                os_matching_records.append(row)

                if behavior_class == TARGET_BEHAVIOR_CLASS:
                    record_dict = {
                        "User ID": user_id,
                        "Device Model": device_model,
                        "Operating System": os_name,
                        "Battery Drain (mAh/day)": battery_drain,
                        "Data Usage (MB/day)": data_usage,
                    }
                    target_behavior_records.append(record_dict)
                    total_battery_drain += battery_drain
                    total_data_usage += data_usage

        except (ValueError, TypeError):
            continue

# Compute Averages
selected_count = len(target_behavior_records)
avg_battery_drain = (total_battery_drain / selected_count) if selected_count > 0 else 0.0
avg_data_usage = (total_data_usage / selected_count) if selected_count > 0 else 0.0

# Export selected users to CSV file
fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

with open(FILTERED_CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_behavior_records)

# Write JSON summary and verify count matching
summary_data = {
    "operating_system": ASSIGNED_OS,
    "user_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": total_records_in_dataset,
    "records_matching_os": len(os_matching_records),
    "records_in_target_behavior_class": selected_count,
    "average_battery_drain_mah_day": round(avg_battery_drain, 2),
    "average_data_usage_mb_day": round(avg_data_usage, 2),
    "exported_csv_file": FILTERED_CSV_PATH.name,
}

with open(SUMMARY_JSON_PATH, mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

count_verified = selected_count == len(target_behavior_records)

# Backup CSV file and get file size
shutil.copy(FILTERED_CSV_PATH, BACKUP_CSV_PATH)
backup_size_bytes = BACKUP_CSV_PATH.stat().st_size if BACKUP_CSV_PATH.exists() else 0

print("=" * 70)
print(f"{'EXERCISE 2: MOBILE DEVICE USAGE PROFILING REPORT':^70}")
print("=" * 70)
print(f"OS & Behavior Class    : {ASSIGNED_OS} | Class {TARGET_BEHAVIOR_CLASS}")
print(f"Total Dataset Records  : {total_records_in_dataset}")
print(f"OS Matching Records    : {len(os_matching_records)}")
print(f"Target Class Records   : {selected_count}")
print(f"Avg Battery Drain      : {avg_battery_drain:.2f} mAh/day")
print(f"Avg Data Usage         : {avg_data_usage:.2f} MB/day")
print(f"Count Verification     : {'VERIFIED' if count_verified else 'FAILED'}")
print(f"Exported CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup File & Size     : {BACKUP_CSV_PATH.name} ({backup_size_bytes} bytes)")
print("=" * 70)

           EXERCISE 2: MOBILE DEVICE USAGE PROFILING REPORT           
OS & Behavior Class    : iOS | Class 3
Total Dataset Records  : 700
OS Matching Records    : 146
Target Class Records   : 32
Avg Battery Drain      : 1492.03 mAh/day
Avg Data Usage         : 828.09 MB/day
Count Verification     : VERIFIED
Exported CSV File      : filtered_user_behavior.csv
JSON Summary File      : user_behavior_summary.json
Backup File & Size     : filtered_user_behavior_backup.csv (1101 bytes)
